# Statistics for Data Analysts/Engineers — Module 2: Visualizing Distributions

In Module 1 we learned to describe data with numbers: mean, median, standard deviation,
IQR. Those numbers are precise, but they can hide the shape of the data -- two very
different datasets can share nearly identical descriptive statistics. In this module
we'll learn to "see" a distribution on a chart instead of only describing it with numbers.

## Table of contents
1. [Why visualization is necessary](#sec1)
2. [Histogram](#sec2)
3. [Box plot](#sec3)
4. [Density plot (KDE)](#sec4)
5. [Skewness and kurtosis](#sec5)
6. [Comparing distributions across groups](#sec6)
7. [Summary and exercises](#sec7)


Every module in this course starts by generating the same synthetic dataset -- thanks to a fixed random seed (`np.random.default_rng(42)`), the numbers always come out identical, so you can open any module independently of the others, even after restarting the kernel.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 400

city = rng.choice(
    ["New York", "Chicago", "Austin", "Seattle", "Denver"],
    size=n, p=[0.30, 0.25, 0.15, 0.15, 0.15],
)
region = rng.choice(["North", "South", "East", "West"], size=n)
age = rng.integers(18, 70, size=n)

# monthly spend: right-skewed distribution (typical for spending data)
monthly_spend = rng.lognormal(mean=5.2, sigma=0.5, size=n).round(2)

# satisfaction correlated with spend + a region effect (deliberately different) + random
# noise, clipped to a 1-10 scale -- this gives Module 10 (ANOVA) a real between-group gap
region_effect = {"North": 0.0, "South": -0.5, "East": 0.3, "West": 0.8}
satisfaction = np.clip(
    monthly_spend / 45
    + np.array([region_effect[r] for r in region])
    + rng.normal(0, 1.5, size=n),
    1, 10,
).round(1)

# A/B campaign group -- with a REAL injected effect on conversion (group B better)
group = rng.choice(["A", "B"], size=n)
conversion_prob = np.where(group == "A", 0.10, 0.20)
converted = rng.binomial(1, conversion_prob)

# spend before/after a loyalty-program rollout (paired data)
spend_before = rng.normal(220, 40, size=n).round(2)
program_effect = rng.normal(15, 10, size=n)
spend_after = (spend_before + program_effect).round(2)

customers = pd.DataFrame({
    "customer_id": np.arange(1, n + 1),
    "city": city,
    "region": region,
    "age": age,
    "monthly_spend": monthly_spend,
    "satisfaction": satisfaction,
    "campaign_group": group,
    "converted": converted,
    "spend_before": spend_before,
    "spend_after": spend_after,
})
customers.head()


<a id="sec1"></a>
## 1. Why visualization is necessary

A famous example from the 1970s (the so-called **Anscombe's quartet**) shows four
completely different datasets that share nearly identical means, variances, and
correlations -- yet look completely different when plotted. The moral: **always plot your
data** before trusting descriptive statistics alone. We'll see this on our own, simpler
examples.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


<a id="sec2"></a>
## 2. Histogram

A **histogram** splits the range of values into equal intervals ("bins") and shows how
many observations fall into each one. It's the most basic way to see the shape of a
distribution -- whether it's symmetric, skewed, unimodal, or multimodal.

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(customers["monthly_spend"], bins=30, edgecolor="white")
plt.xlabel("Monthly spend")
plt.ylabel("Number of customers")
plt.title("Histogram: customer monthly spend")
plt.show()


Now you can clearly see what we only inferred from numbers in Module 1: the
distribution has a long "tail" to the right -- most customers spend relatively little,
and a handful spend a lot. That's exactly **right skewness**, which section 5 covers.

> ⚠️ **The number of bins changes the picture**
>
> Too few bins flattens the histogram and hides structure in the data; too many makes it look like jagged noise. There's no single perfect number; `bins=30` is a reasonable starting point for a few hundred to a few thousand observations, but it's worth experimenting (try swapping in `bins=10` and `bins=100` in the cell above and see the difference).

<a id="sec3"></a>
## 3. Box plot

A **box plot** draws directly what we computed by hand in Module 1: a box from Q1 to Q3
(i.e. the IQR), a line in the middle of the box for the median, "whiskers" reaching to
`Q1 - 1.5*IQR`/`Q3 + 1.5*IQR`, and points beyond the whiskers are exactly the same
outliers we computed by hand.

In [ ]:
plt.figure(figsize=(6, 4))
plt.boxplot(customers["monthly_spend"], vert=False)
plt.xlabel("Monthly spend")
plt.title("Box plot: customer monthly spend")
plt.show()


> 📊 **A box plot is a compressed Module 1**
>
> If you understand a box plot, you understand half of Module 1 in one picture: median, Q1, Q3, IQR, and outliers -- all on one small chart. That's exactly why box plots are so popular for quickly comparing many groups at once (you'll see this in section 6).

<a id="sec4"></a>
## 4. Density plot (KDE)

A **density plot** (*Kernel Density Estimate*, KDE) is a smoothed version of a histogram --
instead of discrete bars it draws a continuous curve approximating the shape of the
distribution. It's often clearer than a histogram, especially when comparing several
distributions on one chart.

In [ ]:
plt.figure(figsize=(8, 4))
sns.kdeplot(customers["monthly_spend"], fill=True)
plt.xlabel("Monthly spend")
plt.title("Density plot: monthly spend")
plt.show()


<a id="sec5"></a>
## 5. Skewness and kurtosis

Since we can "see" skewness by eye, it's worth being able to **measure it as a number**
too:

- **Skewness** — measures the asymmetry of a distribution. `0` is a symmetric
  distribution, a positive value means right skewness (a long tail to the right, exactly
  like our spend data), a negative value means left skewness.
- **Kurtosis** — measures how "fat" a distribution's tails are relative to a normal
  distribution. High kurtosis means more extreme values (fatter tails) than a normal
  distribution.

In [ ]:
from scipy import stats

skewness = stats.skew(customers["monthly_spend"])
kurt = stats.kurtosis(customers["monthly_spend"])

print(f"Skewness: {skewness:.2f}")
print(f"Kurtosis: {kurt:.2f}")


> ⚠️ **scipy.stats.kurtosis returns 'excess' kurtosis**
>
> By default `scipy.stats.kurtosis` returns so-called *excess kurtosis* -- kurtosis minus 3, so that a normal distribution comes out as `0` (not `3`, as in the textbook definition). A positive value = fatter tails than a normal distribution, negative = thinner.

<a id="sec6"></a>
## 6. Comparing distributions across groups

The most common question in practice isn't "what is the distribution of X", but "does the
distribution of X differ between groups A and B?". Box plots and histograms are great for
this once you draw them side by side instead of separately.

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=customers, x="region", y="satisfaction")
plt.xlabel("Region")
plt.ylabel("Satisfaction")
plt.title("Customer satisfaction by region")
plt.show()


You can already see by eye that median satisfaction differs between regions (exactly
the effect we deliberately built into our data). Is that difference "real", or could it
have arisen by chance? That question needs a **statistical test** -- we'll come back to
exactly this kind of question in Module 10 (ANOVA), where we'll test this same regional
difference formally.

In [ ]:
plt.figure(figsize=(8, 4))
for group, data in customers.groupby("campaign_group"):
    sns.kdeplot(data["monthly_spend"], label=f"Group {group}", fill=True, alpha=0.4)
plt.xlabel("Monthly spend")
plt.title("Spend by A/B campaign group")
plt.legend()
plt.show()


<a id="sec7"></a>
## 7. Summary and exercises

In this module we learned to:
- draw histograms and interpret the shape of a distribution,
- draw box plots and connect them to the IQR/outliers from Module 1,
- draw density plots (KDE) as a smoothed alternative to a histogram,
- compute skewness and kurtosis as numeric measures of a distribution's shape,
- compare distributions across groups visually, as a lead-in to formal tests.

> 📝 **Exercise 1: Age histogram**
>
> Draw a histogram of the `age` column with 20 bins. Compare the shape to the spend histogram from section 2 -- is age skewed too?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
plt.figure(figsize=(8, 4))
plt.hist(customers["age"], bins=20, edgecolor="white")
plt.xlabel("Age")
plt.ylabel("Number of customers")
plt.title("Histogram: customer age")
plt.show()
```

Age comes from a uniform distribution, so the histogram should look roughly flat/even -- no pronounced tail, unlike spend.
</details>

> 📝 **Exercise 2: Box plot by city**
>
> Draw a box plot of `monthly_spend` separately for each `city` (like section 6, but for cities instead of regions). Which city has the highest median spend?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
plt.figure(figsize=(8, 4))
sns.boxplot(data=customers, x="city", y="monthly_spend")
plt.xlabel("City")
plt.ylabel("Monthly spend")
plt.title("Monthly spend by city")
plt.show()
```
</details>

> 📝 **Exercise 3: Skewness of satisfaction**
>
> Compute the skewness and kurtosis of the `satisfaction` column. Is the distribution more or less skewed than spend from section 5?

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
from scipy import stats

print(stats.skew(customers["satisfaction"]))
print(stats.kurtosis(customers["satisfaction"]))
```

Satisfaction is built mostly from scaled spend plus normal noise, clipped to the 1-10 range -- expect much lower skewness than raw spend.
</details>

> 📝 **Exercise 4: Density plot by group**
>
> Draw one chart with two density curves (`sns.kdeplot`) of the `satisfaction` column, separately for `campaign_group == 'A'` and `'B'`, on a single chart with a legend.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
plt.figure(figsize=(8, 4))
for group, data in customers.groupby("campaign_group"):
    sns.kdeplot(data["satisfaction"], label=f"Group {group}", fill=True, alpha=0.4)
plt.xlabel("Satisfaction")
plt.title("Satisfaction by campaign group")
plt.legend()
plt.show()
```
</details>

> 🔥 **Challenge: a grid of histograms**
>
> Use `plt.subplots(2, 2, figsize=(10, 8))` to draw four histograms (`monthly_spend`, `age`, `satisfaction`, `spend_after`) in a single 2x2 grid, each with its own title matching the column name. Hint: `axes.flat` lets you iterate over all four axes at once.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
columns = ["monthly_spend", "age", "satisfaction", "spend_after"]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for column, ax in zip(columns, axes.flat):
    ax.hist(customers[column], bins=20, edgecolor="white")
    ax.set_title(column)
plt.tight_layout()
plt.show()
```
</details>

## What's next?

In **Module 3** we'll step back to theory: what probability actually is, how to compute
conditional probability, and what Bayes' theorem tells us. This is the foundation for the
probability distributions in Module 4 and for the rest of inferential statistics.